# Processing 2d-3d dataset
### Matteo Calviello

Working on getting connected to hugging face and getting approval to use the dataset as it is gated.

In [1]:
from huggingface_hub import hf_hub_download, login
import os
import zipfile
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, login
from glob import glob
import random
from IPython.display import display, Image

In [ ]:
# Replace with your actual token
login("you wish")

In [ ]:
def initialize_folders(categories):
    # Creates the nested directory structure.
    paths = ["data/ShapeNet", "data/Sketches"]
    for base in paths:
        for cat in categories:
            os.makedirs(os.path.join(base, cat), exist_ok=True)
    print("[*] Directory structure initialized.")

def download_shapenet_category(repo_id, category_id):
    # Downloads a category ZIP and extracts a specific number of models.
    print(f"[*] Accessing Hugging Face for Category: {category_id}")
    
    # Download the ZIP from the gated repo
    zip_path = hf_hub_download(
        repo_id=repo_id, 
        filename=f"{category_id}.zip", 
        repo_type="dataset"
    )
    
    target_dir = os.path.join("data/ShapeNet", category_id)
    
    print(f"[*] Extracting models to {target_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get all model files in the zip
        all_files = zip_ref.namelist()
        obj_files = [f for f in all_files if f.endswith('model_normalized.obj')]
        
        # Limit extraction so we don't fill up the hard drive during testing
        to_extract = obj_files
        
        for file in to_extract:
            # We extract into our specific data folder
            zip_ref.extract(file, target_dir)
            
    print(f"[*] Successfully downloaded and extracted {len(to_extract)} models.")

### Doanload dataset, limit right now is 50 samples per class but can be changed to get them all

In [ ]:
MY_CATEGORIES = ["02691156", "02828884", "02958343", "03001627", "03636649", "04256520", "04379243", "04530566"] 
REPO = "ShapeNet/ShapeNetCore"

initialize_folders(MY_CATEGORIES)

for cat_id in MY_CATEGORIES:
    download_shapenet_category(REPO, cat_id)

[*] Directory structure initialized.
[*] Accessing Hugging Face for Category: 02691156
[*] Extracting models to data/ShapeNet/02691156...
[*] Successfully downloaded and extracted 50 models.
[*] Accessing Hugging Face for Category: 02828884
[*] Extracting models to data/ShapeNet/02828884...
[*] Successfully downloaded and extracted 50 models.
[*] Accessing Hugging Face for Category: 02958343
[*] Extracting models to data/ShapeNet/02958343...
[*] Successfully downloaded and extracted 50 models.
[*] Accessing Hugging Face for Category: 03001627
[*] Extracting models to data/ShapeNet/03001627...
[*] Successfully downloaded and extracted 50 models.
[*] Accessing Hugging Face for Category: 03636649
[*] Extracting models to data/ShapeNet/03636649...
[*] Successfully downloaded and extracted 50 models.
[*] Accessing Hugging Face for Category: 04256520
[*] Extracting models to data/ShapeNet/04256520...
[*] Successfully downloaded and extracted 50 models.
[*] Accessing Hugging Face for Category

In [4]:
MY_CATEGORIES = ["02691156", "02828884", "02958343", "03001627", "03636649", "04256520", "04379243", "04530566"] 
BASE_PATH = "data/ShapeNet"
category_names = {
    "02691156": "Airplane", "02828884": "Bench", "02958343": "Car", 
    "03001627": "Chair", "03636649": "Lamp", "04256520": "Sofa", 
    "04379243": "Table", "04530566": "Watercraft"
}

In [5]:
print(f"{'Category':<15} | {'ID':<10} | {'Count':<6}")
print("-" * 35)
    
for cat_id in MY_CATEGORIES:
    path = os.path.join(BASE_PATH, cat_id)
        
    # Look for model_normalized.obj files recursively
    models = glob(os.path.join(path, "**", "model_normalized.obj"), recursive=True)
    count = len(models)
        
    name = category_names.get(cat_id, "Unknown")
    print(f"{name:<15} | {cat_id:<10} | {count:<6}")

Category        | ID         | Count 
-----------------------------------
Airplane        | 02691156   | 50    
Bench           | 02828884   | 50    
Car             | 02958343   | 50    
Chair           | 03001627   | 50    
Lamp            | 03636649   | 50    
Sofa            | 04256520   | 50    
Table           | 04379243   | 50    
Watercraft      | 04530566   | 50    


### Print sample of each class to check what they look like

In [ ]:
viewers = []

for cat_id in MY_CATEGORIES:
    path = os.path.join(BASE_PATH, cat_id)
    models = glob(os.path.join(path, "**", "model_normalized.obj"), recursive=True)
    
    if models:
        # Load a random model
        mesh = trimesh.load(random.choice(models))
        scene = mesh.scene() if not isinstance(mesh, trimesh.Scene) else mesh
        
        # Increased width to 600px for a better vertical view
        out = widgets.Output(layout={
            'border': '2px solid #444', 
            'width': '600px', 
            'height': '400px',
            'margin': '10px auto'
        })
        
        with out:
            print(f"--- CATEGORY: {category_names[cat_id].upper()} ---")
            display(scene.show(viewer='jupyter'))
        viewers.append(out)

grid = widgets.GridBox(viewers, layout=widgets.Layout(
    grid_template_columns="repeat(1, 100%)",
    align_items='center'
))

display(grid)

GridBox(children=(Output(layout=Layout(border_bottom='2px solid #444', border_left='2px solid #444', border_ri…

### Get some more randoms per one class to see how they look like

In [ ]:
# Configuration
AIRPLANE_ID = "02691156"
path = os.path.join(BASE_PATH, AIRPLANE_ID)
models = glob(os.path.join(path, "**", "model_normalized.obj"), recursive=True)

viewers = []

if len(models) >= 5:
    sample_subset = random.sample(models, 5)
    
    for i, model_path in enumerate(sample_subset):
        mesh = trimesh.load(model_path)
        scene = mesh.scene() if not isinstance(mesh, trimesh.Scene) else mesh
        
        # Create a clean container for each airplane
        out = widgets.Output(layout={
            'border': '2px solid #2196F3', # Blue border for Airplanes
            'width': '600px', 
            'height': '400px',
            'margin': '10px auto'
        })
        
        with out:
            print(f"--- AIRPLANE SAMPLE {i+1} ---")
            print(f"Path: ...{model_path[-50:]}")
            display(scene.show(viewer='jupyter'))
        viewers.append(out)

    grid = widgets.GridBox(viewers, layout=widgets.Layout(
        grid_template_columns="repeat(1, 100%)",
        align_items='center'
    ))

    display(grid)
else:
    print(f"[!] Not enough models found. Found: {len(models)}")

GridBox(children=(Output(layout=Layout(border_bottom='2px solid #2196F3', border_left='2px solid #2196F3', bor…

### testing generation of 2d sketch

In [ ]:
def generate_sketch_vs_ground_truth(model_path, num_points=10000):
    # Load and Flatten
    loaded = trimesh.load(model_path)
    mesh = loaded.to_mesh() if isinstance(loaded, trimesh.Scene) else loaded
    
    # do some robust normalization - If volume is zero or invalid, use bounding box center, this helped prevent RuntimeWarnings
    if mesh.volume > 1e-8:
        center = mesh.center_mass
    else:
        center = mesh.bounding_box.centroid
    
    mesh.vertices -= center
    
    # Scale to unit sphere
    max_dist = np.max(np.linalg.norm(mesh.vertices, axis=1))
    if max_dist > 0:
        mesh.vertices /= max_dist

    # create sketch by sampling points
    points = mesh.sample(num_points)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(points[:, 0], points[:, 1], s=0.01, c='black', alpha=0.4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title("Generated 2D Sketch (Input)")

    # create the layout with two outputs side by side
    out_2d = widgets.Output(layout={'width': '50%', 'border': '1px solid gray'})
    with out_2d:
        display(fig)
        plt.close(fig)

    # Right Side: The Interactive Ground Truth Mesh
    out_3d = widgets.Output(layout={'width': '50%', 'border': '1px solid gray'})
    with out_3d:
        print("3D Ground Truth Mesh (Target)")
        display(loaded.show(viewer='jupyter'))

    print(f"[*] Comparing Pair for: {os.path.basename(model_path)}")
    display(widgets.HBox([out_2d, out_3d]))

# Test on a random model from airplanes
if models:
    sample = random.choice(models)
    generate_sketch_vs_ground_truth(sample)

[*] Comparing Pair for: model_normalized.obj


### Code to get 2d sketches for all 3d .obj files

In [ ]:
def generate_all_sketches(categories, num_points=50000):
    for cat_id in categories:
        # Get 3D models we downloaded
        src_path = os.path.join("data/ShapeNet", cat_id)
        obj_paths = glob(os.path.join(src_path, "**", "model_normalized.obj"), recursive=True)
        
        print(f"[*] Category {cat_id}: Found {len(obj_paths)} models.")
        
        for i, model_path in enumerate(obj_paths):
            # Create a unique filename for the sketch based on the model's folder ID
            model_id = model_path.split(os.sep)[-3] # Gets the unique ShapeNet ID
            save_path = os.path.join("data/Sketches", cat_id, f"{model_id}.png")
            
            # CHECKPOINT: Skip if the sketch already exists
            if os.path.exists(save_path):
                continue

            try:
                # Load and Normalize
                loaded = trimesh.load(model_path)
                mesh = loaded.to_mesh() if isinstance(loaded, trimesh.Scene) else loaded
                
                # Robust centering (fixes the RuntimeWarnings)
                center = mesh.center_mass if mesh.volume > 1e-8 else mesh.bounding_box.centroid
                mesh.vertices -= center
                
                # Unit sphere scaling
                scale = np.max(np.linalg.norm(mesh.vertices, axis=1))
                if scale > 0: mesh.vertices /= scale

                # High-Density Sampling
                points = mesh.sample(num_points)

                # Render to File - We use 224x224 because it's the standard input size for many ML models
                fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
                ax.scatter(points[:, 0], points[:, 1], s=0.01, c='black', alpha=0.4)
                ax.set_aspect('equal')
                ax.axis('off')
                
                # Save the sketch
                plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
                plt.close(fig)

                if (i + 1) % 50 == 0:
                    print(f"    Processed {i + 1}/{len(obj_paths)} sketches...")

            except Exception as e:
                print(f" [!] Error on {model_id}: {e}")

# This will run for ALL categories you defined earlier
generate_all_sketches(MY_CATEGORIES)

[*] Category 02691156: Found 50 models.
    Processed 50/50 sketches...
[*] Category 02828884: Found 50 models.
    Processed 50/50 sketches...
[*] Category 02958343: Found 50 models.
    Processed 50/50 sketches...
[*] Category 03001627: Found 50 models.
    Processed 50/50 sketches...
[*] Category 03636649: Found 50 models.
    Processed 50/50 sketches...
[*] Category 04256520: Found 50 models.
    Processed 50/50 sketches...
[*] Category 04379243: Found 50 models.
    Processed 50/50 sketches...
[*] Category 04530566: Found 50 models.
    Processed 50/50 sketches...
